# 01 — Data Ingestion & Cleaning

**Objective:** Load the raw DHS subnational malaria indicators dataset for Nigeria, filter to relevant indicators and state-level records, pivot from long to wide format, and save a clean processed file for use in subsequent notebooks.

**Input:** `data/raw/select-malaria-indicators_subnational_nga.csv`  
**Output:** `data/processed/malaria_indicators_wide.csv`

## 1. Imports

In [1]:
import pandas as pd
import numpy as np
import os

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 60)
pd.set_option('display.float_format', '{:.2f}'.format)

## 2. Load Raw Data

In [2]:
RAW_PATH = '../data/raw/select-malaria-indicators_subnational_nga.csv'
PROCESSED_PATH = '../data/processed/malaria_indicators_wide.csv'

df_raw = pd.read_csv(RAW_PATH)

print(f'Raw dataset shape: {df_raw.shape}')
print(f'\nColumns: {df_raw.columns.tolist()}')

Raw dataset shape: (3040, 30)

Columns: ['ISO3', 'Location', 'DataId', 'Indicator', 'Value', 'Precision', 'DHS_CountryCode', 'CountryName', 'SurveyYear', 'SurveyId', 'IndicatorId', 'IndicatorOrder', 'IndicatorType', 'CharacteristicId', 'CharacteristicOrder', 'CharacteristicCategory', 'CharacteristicLabel', 'ByVariableId', 'ByVariableLabel', 'IsTotal', 'IsPreferred', 'SDRID', 'RegionId', 'SurveyYearLabel', 'SurveyType', 'DenominatorWeighted', 'DenominatorUnweighted', 'CILow', 'CIHigh', 'LevelRank']


In [3]:
# Quick overview
df_raw.head(3)

,ISO3,Location,DataId,Indicator,Value,Precision,DHS_CountryCode,CountryName,SurveyYear,SurveyId,IndicatorId,IndicatorOrder,IndicatorType,CharacteristicId,CharacteristicOrder,CharacteristicCategory,CharacteristicLabel,ByVariableId,ByVariableLabel,IsTotal,IsPreferred,SDRID,RegionId,SurveyYearLabel,SurveyType,DenominatorWeighted,DenominatorUnweighted,CILow,CIHigh,LevelRank
0,NGA,North Central,6408275,Households with at least one insecticide-treat...,4.30,1,NG,Nigeria,2003,NG2003DHS,ML_NETP_H_ITN,124661020,I,433005,1433010,Region,North Central,0,NaN,0,1,MLNETPHITN,NGDHS2003433005,2003,DHS,1040,1158,NaN,NaN,1
1,NGA,North East,4197672,Households with at least one insecticide-treat...,1.30,1,NG,Nigeria,2003,NG2003DHS,ML_NETP_H_ITN,124661020,I,433006,1433020,Region,North East,0,NaN,0,1,MLNETPHITN,NGDHS2003433006,2003,DHS,1185,1191,NaN,NaN,1
2,NGA,North West,5726356,Households with at least one insecticide-treat...,3.10,1,NG,Nigeria,2003,NG2003DHS,ML_NETP_H_ITN,124661020,I,433007,1433030,Region,North West,0,NaN,0,1,MLNETPHITN,NGDHS2003433007,2003,DHS,1911,1576,NaN,NaN,1


## 3. Understand the Structure

The raw data is in **long format**: each row is one indicator value for one location in one survey year. We need to understand what locations, years, and indicators are available before we filter.

In [4]:
print('=== SURVEY YEARS ===')
print(sorted(df_raw['SurveyYear'].unique()))

print('\n=== UNIQUE LOCATIONS ===')
print(sorted(df_raw['Location'].unique()))

print('\n=== UNIQUE INDICATORS ===')
for i, ind in enumerate(df_raw['Indicator'].unique(), 1):
    print(f'{i:2}. {ind}')

print('\n=== CHARACTERISTIC CATEGORIES ===')
print(df_raw['CharacteristicCategory'].unique())

=== SURVEY YEARS ===
[np.int64(2003), np.int64(2008), np.int64(2010), np.int64(2013), np.int64(2015), np.int64(2018), np.int64(2021), np.int64(2024)]

=== UNIQUE LOCATIONS ===
['Abia', 'Adamawa', 'Akwa Ibom', 'Anambra', 'Bauchi', 'Bayelsa', 'Benue', 'Borno', 'Cross River', 'Delta', 'Ebonyi', 'Edo', 'Ekiti', 'Enugu', 'FCT Abuja', 'Gombe', 'Imo', 'Jigawa', 'Kaduna', 'Kano', 'Katsina', 'Kebbi', 'Kogi', 'Kwara', 'Lagos', 'Nasarawa', 'Niger', 'North Central', 'North East', 'North West', 'Ogun', 'Ondo', 'Osun', 'Oyo', 'Plateau', 'Rivers', 'Sokoto', 'South East', 'South South', 'South West', 'Taraba', 'Yobe', 'Zamfara']

=== UNIQUE INDICATORS ===
 1. Households with at least one insecticide-treated mosquito net (ITN)
 2. Households with at least one insecticide-treated mosquito net (ITN) for every two persons who stayed in the household the previous night
 3. Persons with access to an insecticide-treated mosquito net (ITN)
 4. Population who slept under an insecticide-treated mosquito net (IT

## 4. Filter: States Only

The dataset contains both state-level rows and regional aggregates (North Central, North East, etc.). We want **state-level rows only** — the 36 states + FCT Abuja.

Regional aggregates are useful for context but would introduce data leakage if used as model inputs (they're derived from the state rows below them).

In [5]:
# Regional aggregates to exclude
REGIONAL_AGGREGATES = [
    'North Central', 'North East', 'North West',
    'South East', 'South South', 'South West'
]

df_states = df_raw[~df_raw['Location'].isin(REGIONAL_AGGREGATES)].copy()

print(f'After removing regional aggregates: {df_states.shape[0]} rows')
print(f'Unique state locations: {df_states["Location"].nunique()}')
print(f'States: {sorted(df_states["Location"].unique())}')

After removing regional aggregates: 2401 rows
Unique state locations: 37
States: ['Abia', 'Adamawa', 'Akwa Ibom', 'Anambra', 'Bauchi', 'Bayelsa', 'Benue', 'Borno', 'Cross River', 'Delta', 'Ebonyi', 'Edo', 'Ekiti', 'Enugu', 'FCT Abuja', 'Gombe', 'Imo', 'Jigawa', 'Kaduna', 'Kano', 'Katsina', 'Kebbi', 'Kogi', 'Kwara', 'Lagos', 'Nasarawa', 'Niger', 'Ogun', 'Ondo', 'Osun', 'Oyo', 'Plateau', 'Rivers', 'Sokoto', 'Taraba', 'Yobe', 'Zamfara']


## 5. Select Relevant Columns

The raw dataset has 30 columns. We only need a subset for our model.

**Keep:**
- `Location` — state name (our unit of analysis)
- `SurveyYear` — year of the DHS/MIS survey
- `SurveyType` — DHS or MIS (useful metadata)
- `Indicator` — the indicator name (becomes column names after pivot)
- `Value` — the indicator value
- `CILow`, `CIHigh` — confidence intervals (useful for the reliability module)
- `DenominatorWeighted` — sample size (also useful for reliability)

In [6]:
COLS_TO_KEEP = [
    'Location', 'SurveyYear', 'SurveyType',
    'Indicator', 'Value', 'CILow', 'CIHigh', 'DenominatorWeighted'
]

df_slim = df_states[COLS_TO_KEEP].copy()
print(f'Slim dataset shape: {df_slim.shape}')
df_slim.head()

Slim dataset shape: (2401, 8)


,Location,SurveyYear,SurveyType,Indicator,Value,CILow,CIHigh,DenominatorWeighted
220,FCT Abuja,2013,DHS,Households with at least one insecticide-treat...,27.40,NaN,NaN,361
221,Benue,2013,DHS,Households with at least one insecticide-treat...,73.20,NaN,NaN,1365
222,Kogi,2013,DHS,Households with at least one insecticide-treat...,25.50,NaN,NaN,876
223,Kwara,2013,DHS,Households with at least one insecticide-treat...,34.00,NaN,NaN,617
224,Nasarawa,2013,DHS,Households with at least one insecticide-treat...,52.70,NaN,NaN,550


## 6. Create Short Indicator Names

The full indicator names are long strings — they will become messy column names after pivoting. We'll map them to clean short codes.

In [7]:
INDICATOR_MAP = {
    'Households with at least one insecticide-treated mosquito net (ITN)': 'itn_hh_any',
    'Households with at least one insecticide-treated mosquito net (ITN) for every two persons who stayed in the household the previous night': 'itn_hh_ratio',
    'Persons with access to an insecticide-treated mosquito net (ITN)': 'itn_access',
    'Population who slept under an insecticide-treated mosquito net (ITN) last night': 'itn_slept_pop',
    'Children under 5 who slept under an insecticide-treated net (ITN)': 'itn_slept_u5',
    'Children under 5 who slept under an insecticide-treated mosquito net (ITN) last night of those living in a household with at least one ITN': 'itn_slept_u5_conditional',
    'Pregnant women who slept under an insecticide-treated net (ITN)': 'itn_slept_preg',
    'Pregnant women who slept under insecticide-treated mosquito net (ITN) last night of those living in a household with at least one ITN': 'itn_slept_preg_conditional',
    'SP/Fansidar 3+ doses during pregnancy': 'sp_3plus_doses',
    'SP/Fansidar 2+ doses, at least one during ANC visit': 'sp_2plus_anc',
    'Children with fever for whom advice or treatment was sought': 'fever_treatment_sought',
    'Children who took any ACT': 'act_treatment',
    'Children with hemoglobin lower than 8.0 g/dl': 'anaemia_severe',
    'Malaria prevalence according to RDT': 'prevalence_rdt',
    'Malaria prevalence according to microscopy': 'prevalence_microscopy'
}

df_slim['indicator_code'] = df_slim['Indicator'].map(INDICATOR_MAP)

# Check for any unmapped indicators
unmapped = df_slim[df_slim['indicator_code'].isna()]['Indicator'].unique()
if len(unmapped) > 0:
    print(f'WARNING: Unmapped indicators: {unmapped}')
else:
    print('All indicators mapped successfully.')

df_slim.head()

All indicators mapped successfully.


,Location,SurveyYear,SurveyType,Indicator,Value,CILow,CIHigh,DenominatorWeighted,indicator_code
220,FCT Abuja,2013,DHS,Households with at least one insecticide-treat...,27.40,NaN,NaN,361,itn_hh_any
221,Benue,2013,DHS,Households with at least one insecticide-treat...,73.20,NaN,NaN,1365,itn_hh_any
222,Kogi,2013,DHS,Households with at least one insecticide-treat...,25.50,NaN,NaN,876,itn_hh_any
223,Kwara,2013,DHS,Households with at least one insecticide-treat...,34.00,NaN,NaN,617,itn_hh_any
224,Nasarawa,2013,DHS,Households with at least one insecticide-treat...,52.70,NaN,NaN,550,itn_hh_any


## 7. Pivot to Wide Format

Transform from long format (one row per indicator) to wide format (one row per state × survey year, one column per indicator). This is the structure our model needs.

In [8]:
df_wide = df_slim.pivot_table(
    index=['Location', 'SurveyYear', 'SurveyType'],
    columns='indicator_code',
    values='Value',
    aggfunc='mean'  # some state-year combinations may have slight duplicates
).reset_index()

# Flatten column names
df_wide.columns.name = None

print(f'Wide dataset shape: {df_wide.shape}')
print(f'Rows = state × survey year combinations')
df_wide.head(10)

Wide dataset shape: (185, 18)
Rows = state × survey year combinations


,Location,SurveyYear,SurveyType,act_treatment,anaemia_severe,fever_treatment_sought,itn_access,itn_hh_any,itn_hh_ratio,itn_slept_pop,itn_slept_preg,itn_slept_preg_conditional,itn_slept_u5,itn_slept_u5_conditional,prevalence_microscopy,prevalence_rdt,sp_2plus_anc,sp_3plus_doses
0,Abia,2013,DHS,NaN,NaN,69.20,46.00,59.60,34.90,18.70,18.90,33.60,23.50,34.10,NaN,NaN,21.70,19.70
1,Abia,2015,MIS,NaN,1.40,78.80,40.10,51.90,29.90,7.00,NaN,NaN,8.20,16.00,8.20,21.10,36.00,26.60
2,Abia,2018,DHS,NaN,2.80,86.60,32.20,45.40,22.90,22.70,25.20,51.40,23.90,48.10,13.50,20.70,76.90,73.70
3,Abia,2021,MIS,89.50,6.60,79.10,24.10,30.30,15.00,15.30,NaN,NaN,21.90,63.30,14.50,26.40,78.20,51.10
4,Abia,2024,DHS,82.40,2.50,60.60,19.60,25.40,13.30,15.20,12.10,NaN,20.00,57.60,NaN,NaN,66.70,25.70
5,Adamawa,2013,DHS,NaN,NaN,81.10,54.20,72.80,35.20,10.80,13.10,16.80,12.60,15.80,NaN,NaN,22.30,12.20
6,Adamawa,2015,MIS,30.90,11.40,55.50,48.80,70.60,31.10,31.60,35.70,40.50,32.40,50.00,34.70,55.50,31.30,22.90
7,Adamawa,2018,DHS,67.50,6.70,49.30,45.20,47.70,32.70,42.60,37.80,89.40,46.80,90.00,21.10,38.90,36.70,4.40
8,Adamawa,2021,MIS,NaN,4.90,65.60,73.90,90.20,55.90,67.50,72.80,86.10,78.10,83.50,10.70,28.00,53.90,41.90
9,Adamawa,2024,DHS,44.10,6.50,70.30,66.00,78.60,48.90,45.20,42.60,54.30,47.90,60.70,NaN,NaN,53.50,34.40


## 8. Inspect Missingness

Not every state has data for every indicator in every survey year — this is expected with DHS data. We need to understand the pattern before deciding how to handle it in feature engineering.

In [9]:
print('=== MISSING VALUES PER INDICATOR ===')
missing = df_wide.isnull().sum()
missing_pct = (df_wide.isnull().sum() / len(df_wide) * 100).round(1)
missing_df = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
print(missing_df[missing_df['missing_count'] > 0].sort_values('missing_pct', ascending=False))

=== MISSING VALUES PER INDICATOR ===
                            missing_count  missing_pct
act_treatment                          75        40.50
prevalence_microscopy                  74        40.00
prevalence_rdt                         74        40.00
itn_slept_preg_conditional             65        35.10
itn_slept_preg                         39        21.10
anaemia_severe                         37        20.00
fever_treatment_sought                 10         5.40


In [10]:
print('=== ROWS PER SURVEY YEAR ===')
print(df_wide.groupby('SurveyYear')['Location'].count().rename('state_count'))

print('\n=== PREVALENCE RDT COVERAGE BY YEAR ===')
print(df_wide.groupby('SurveyYear')['prevalence_rdt'].agg(['count', 'mean', 'min', 'max']).round(2))

=== ROWS PER SURVEY YEAR ===
SurveyYear
2013    37
2015    37
2018    37
2021    37
2024    37
Name: state_count, dtype: int64

=== PREVALENCE RDT COVERAGE BY YEAR ===
            count  mean  min   max
SurveyYear                        
2013            0   NaN  NaN   NaN
2015           37 41.49 1.90 69.90
2018           37 36.38 3.40 76.80
2021           37 35.12 3.20 75.60
2024            0   NaN  NaN   NaN


## 9. Flag Our Target Variable

Our target for the risk classification model is `prevalence_rdt` — malaria prevalence according to Rapid Diagnostic Test (RDT), measured in children under 5.

We'll define three risk classes:
- **Low**: prevalence < 10%
- **Medium**: prevalence 10–30%
- **High**: prevalence > 30%

These thresholds align with Nigeria's National Malaria Strategic Plan stratification bands.

In [11]:
def assign_risk_class(prevalence):
    if pd.isna(prevalence):
        return np.nan
    elif prevalence < 10:
        return 'Low'
    elif prevalence <= 30:
        return 'Medium'
    else:
        return 'High'

df_wide['risk_class'] = df_wide['prevalence_rdt'].apply(assign_risk_class)

print('=== RISK CLASS DISTRIBUTION ===')
print(df_wide['risk_class'].value_counts(dropna=False))

print('\n=== RISK CLASS BY SURVEY YEAR ===')
print(df_wide.groupby('SurveyYear')['risk_class'].value_counts().unstack(fill_value=0))

=== RISK CLASS DISTRIBUTION ===
risk_class
High      76
NaN       74
Medium    31
Low        4
Name: count, dtype: int64

=== RISK CLASS BY SURVEY YEAR ===
risk_class  High  Low  Medium
SurveyYear                   
2015          27    2       8
2018          27    1       9
2021          22    1      14


## 10. Add a Reliability Proxy Column

As a first-pass reliability signal (to be developed fully in notebook 05), we flag rows where the primary target (`prevalence_rdt`) is missing — these are state-years where no RDT data was collected, making the risk estimate unreliable.

In [12]:
df_wide['rdt_data_available'] = df_wide['prevalence_rdt'].notna().astype(int)

print('RDT data available:', df_wide['rdt_data_available'].sum())
print('RDT data missing: ', (df_wide['rdt_data_available'] == 0).sum())

RDT data available: 111
RDT data missing:  74


## 11. Save Processed Data

In [13]:
os.makedirs('../data/processed', exist_ok=True)
df_wide.to_csv(PROCESSED_PATH, index=False)

print(f'Saved to {PROCESSED_PATH}')
print(f'Final shape: {df_wide.shape}')
print(f'Columns: {df_wide.columns.tolist()}')

Saved to ../data/processed/malaria_indicators_wide.csv
Final shape: (185, 20)
Columns: ['Location', 'SurveyYear', 'SurveyType', 'act_treatment', 'anaemia_severe', 'fever_treatment_sought', 'itn_access', 'itn_hh_any', 'itn_hh_ratio', 'itn_slept_pop', 'itn_slept_preg', 'itn_slept_preg_conditional', 'itn_slept_u5', 'itn_slept_u5_conditional', 'prevalence_microscopy', 'prevalence_rdt', 'sp_2plus_anc', 'sp_3plus_doses', 'risk_class', 'rdt_data_available']


## 12. Summary

| Step | Action | Result |
|------|--------|--------|
| Load | Read raw CSV | 3,040 rows × 30 columns |
| Filter | Remove regional aggregates | State-level rows only |
| Slim | Keep 8 relevant columns | Reduced schema |
| Map | Rename indicators to short codes | 15 indicator codes |
| Pivot | Long → wide format | One row per state × survey year |
| Target | Assign risk class from RDT prevalence | Low / Medium / High |
| Save | Write to `data/processed/` | Ready for EDA |

**Next notebook:** `02_eda.ipynb` — Exploratory Data Analysis